# 02 — Prompt Engineering en práctica

> **Bloque:** LLMs | **Nivel:** Introductorio-Intermedio
>
> Notebook complementario al tutorial [02-prompt-engineering.md](../../llms/02-prompt-engineering.md)

Experimentaremos con las técnicas de prompt engineering directamente sobre Claude, midiendo el impacto de cada técnica.

In [ ]:
import anthropic
import os
import json
import time

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'  # Haiku para experimentos: rápido y económico

def llamar(prompt: str, system: str = None, temperature: float = 0.0,
           max_tokens: int = 400) -> str:
    """Helper para llamar a Claude de forma concisa."""
    kwargs = dict(
        model=MODELO,
        max_tokens=max_tokens,
        temperature=temperature,
        messages=[{'role': 'user', 'content': prompt}]
    )
    if system:
        kwargs['system'] = system
    return client.messages.create(**kwargs).content[0].text

print('✓ Configuración lista. Modelo:', MODELO)

## 1. Zero-shot vs Few-shot

Comparamos el mismo problema con y sin ejemplos.

In [ ]:
texto_test = 'La aplicación carga bien pero el botón de guardar a veces no responde'

# Zero-shot
prompt_zero = f"""Clasifica este texto en: BUG, FEATURE_REQUEST, PREGUNTA o QUEJA.
Responde solo con la categoría.

Texto: \"{texto_test}\""""

# Few-shot
prompt_few = f"""Clasifica estos textos:

Texto: "El botón de login no funciona" → BUG
Texto: "¿Puedo exportar en formato Excel?" → PREGUNTA
Texto: "Sería útil tener modo oscuro" → FEATURE_REQUEST
Texto: "Llevo esperando respuesta 3 días" → QUEJA

Texto: \"{texto_test}\" →"""

r_zero = llamar(prompt_zero)
r_few = llamar(prompt_few)

print(f'Texto: "{texto_test}"')
print(f'\nZero-shot: {r_zero}')
print(f'Few-shot:  {r_few}')

## 2. Chain-of-Thought

El pensamiento paso a paso mejora el razonamiento.

In [ ]:
problema = """Una empresa tiene 3 planes de IA:
- Básico: 1.000 llamadas/mes a 0.002€/llamada
- Pro: 5.000 llamadas/mes a 0.0015€/llamada
- Enterprise: ilimitado por 150€/mes

Si usan 4.200 llamadas al mes, ¿qué plan les conviene más?"""

# Sin CoT
r_sin_cot = llamar(f"{problema}\n\n¿Qué plan es mejor?")

# Con CoT
r_con_cot = llamar(f"{problema}\n\nPiensa paso a paso y calcula el coste de cada opción antes de responder.",
                   max_tokens=600)

print('=== SIN Chain-of-Thought ===')
print(r_sin_cot)
print('\n=== CON Chain-of-Thought ===')
print(r_con_cot)

## 3. Efecto de la temperatura

In [ ]:
import matplotlib.pyplot as plt

prompt_creativo = 'Completa esta frase de forma original: "La inteligencia artificial es como..."'
temperaturas = [0.0, 0.3, 0.7, 1.0]

print('Mismo prompt, diferentes temperaturas:\n')
for temp in temperaturas:
    respuesta = llamar(prompt_creativo, temperature=temp, max_tokens=80)
    print(f'T={temp}: {respuesta.strip()}')
    print()

## 4. System prompt y rol

In [ ]:
pregunta = '¿Qué es el machine learning?'

roles = {
    'Sin system prompt': None,
    'Experto técnico': 'Eres un investigador senior de ML. Usa terminología técnica precisa.',
    'Profesor de primaria': 'Eres un profesor que explica conceptos a niños de 10 años. Usa analogías simples y lenguaje muy sencillo.',
    'Ejecutivo de negocio': 'Eres un consultor de negocios. Enfócate en el valor empresarial y evita la jerga técnica.',
}

for rol, system in roles.items():
    respuesta = llamar(pregunta, system=system, max_tokens=150, temperature=0.3)
    print(f'\n--- {rol} ---')
    print(respuesta)

## 5. Output estructurado

In [ ]:
texto_empresa = """
Anthropic es una empresa de seguridad en IA fundada en 2021 por Dario Amodei y Daniela Amodei,
junto con otros ex-empleados de OpenAI. Tiene sede en San Francisco, California.
Es conocida por desarrollar Claude, su familia de modelos de lenguaje. 
Ha recibido inversiones significativas de Google y Amazon.
"""

prompt = f"""Extrae información de este texto y devuelve SOLO un JSON válido:
{{
  "nombre": "",
  "fundacion_año": 0,
  "fundadores": [],
  "sede": "",
  "producto_principal": "",
  "inversores": []
}}

Texto:
{texto_empresa}"""

respuesta = llamar(prompt, max_tokens=300)

try:
    datos = json.loads(respuesta)
    print('JSON parseado correctamente:')
    print(json.dumps(datos, ensure_ascii=False, indent=2))
except json.JSONDecodeError:
    print('Respuesta sin parsear:')
    print(respuesta)

## 6. Experimento libre

Diseña tu propio prompt y experimenta con las técnicas aprendidas.

In [ ]:
# Tu experimento aquí
mi_prompt = 'Escribe 3 casos de uso de IA en el sector legal'
mi_system = 'Eres un abogado especialista en tecnología. Responde en formato de lista numerada.'

resultado = llamar(mi_prompt, system=mi_system, temperature=0.5, max_tokens=400)
print(resultado)

---
**Tutorial relacionado:** [02-prompt-engineering.md](../../llms/02-prompt-engineering.md)